# Tổng hợp: Optimization & Overfitting trong MLP (từ Scratch)

Notebook này gộp **hai trụ cột quan trọng nhất** khi huấn luyện mạng nơ-ron, dùng chung **một lớp `MLP`** hiện thực từ đầu bằng **NumPy**:

| Trụ cột | Câu hỏi trả lời | Công cụ |
| :--- | :--- | :--- |
| **1. Optimization (Tối ưu)** | Làm sao học **NHANH** và hội tụ tốt? | SGD, Momentum, Nesterov, AdaGrad, RMSprop, Adam, AdamW + Mini-batch, Gradient Clipping, LR Scheduler, Warmup |
| **2. Regularization (Chống Overfit)** | Làm sao **TỔNG QUÁT HÓA** tốt trên dữ liệu mới? | L1, L2, Dropout, Data Augmentation, Early Stopping, LR Decay |

**Ý tưởng cốt lõi:**
- **Optimizer** quyết định *tốc độ và chất lượng hội tụ* trên tập Train.
- **Regularization** quyết định *khoảng cách Train ↔ Test* (chống học vẹt nhiễu).
- Một mô hình tốt cần **CẢ HAI**: học nhanh **và** không overfit.

**Bố cục:**
- **Phần A** — So sánh Optimizers (không regularization).
- **Phần B** — Ablation Regularization (cố định optimizer = Adam).
- **Phần C** — Kết hợp: Optimizer tốt + Regularization đầy đủ vs Baseline thô sơ.

In [ ]:
# Bước 1: Import thư viện
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Đã import thành công các thư viện!")

## Lớp `MLP` hợp nhất — Optimizer + Regularization

Một lớp duy nhất gộp toàn bộ:

**Optimization:**
- 7 optimizers: `sgd`, `momentum`, `nesterov`, `adagrad`, `rmsprop`, `adam`, `adamw`
- Mini-batch training, Gradient Clipping, LR Scheduler (`cosine`/`step`), Warmup

**Regularization:**
- `l1_lambda`, `l2_lambda` (L2 = weight decay; với AdamW tách rời theo kiểu *decoupled*)
- `dropout_rate` (Inverted Dropout), `patience` (Early Stopping theo Val Loss)
- `augment` (thêm nhiễu Gaussian vào input mỗi epoch)

Mỗi epoch ghi lại Train/Val **Loss** và **Accuracy** để vừa xem *hội tụ* (optimization) vừa xem *Gap* (overfitting).

In [ ]:
# Bước 2: Định nghĩa lớp MLP hợp nhất (Optimizer + Regularization) từ Scratch

class MLP:
    """MLP từ scratch: gộp 7 Optimizer + 6 kỹ thuật Regularization + Mini-batch/Clipping/Scheduler/Warmup."""

    def __init__(
        self,
        layer_sizes: list[int],
        learning_rate: float = 0.01,
        epochs: int = 1000,
        batch_size: int = 32,
        optimizer: str = "adam",
        beta1: float = 0.9,
        beta2: float = 0.999,
        epsilon: float = 1e-8,
        # --- Regularization ---
        l1_lambda: float = 0.0,
        l2_lambda: float = 0.0,
        dropout_rate: float = 0.0,
        patience: int = None,
        # --- Optimization nâng cao ---
        clip_norm: float = None,
        lr_scheduler: str = None,   # "cosine", "step", hoặc None
        warmup_steps: int = 0,
    ):
        self.layer_sizes = layer_sizes
        self.init_lr = learning_rate
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.optimizer = optimizer.lower()
        self.beta1, self.beta2, self.epsilon = beta1, beta2, epsilon
        self.l1_lambda, self.l2_lambda = l1_lambda, l2_lambda
        self.dropout_rate = dropout_rate
        self.patience = patience
        self.clip_norm = clip_norm
        self.lr_scheduler = lr_scheduler
        self.warmup_steps = warmup_steps

        # He Initialization
        self.W, self.b = [], []
        for i in range(1, len(layer_sizes)):
            limit = np.sqrt(2.0 / layer_sizes[i - 1])
            self.W.append(np.random.randn(layer_sizes[i], layer_sizes[i - 1]) * limit)
            self.b.append(np.zeros((layer_sizes[i], 1)))

        # Trạng thái optimizer (m: moment1, v: moment2/velocity, s: adaptive)
        self.m_W = [np.zeros_like(w) for w in self.W]
        self.m_b = [np.zeros_like(bias) for bias in self.b]
        self.v_W = [np.zeros_like(w) for w in self.W]
        self.v_b = [np.zeros_like(bias) for bias in self.b]
        self.s_W = [np.zeros_like(w) for w in self.W]
        self.s_b = [np.zeros_like(bias) for bias in self.b]

        self.dropout_masks = []
        self.training = True

        # Lịch sử
        self.loss_history, self.val_loss_history = [], []
        self.accuracy_history, self.val_accuracy_history = [], []
        self.best_W, self.best_b, self.best_epoch = [], [], 0

    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def _sigmoid_derivative(a):
        return a * (1.0 - a)

    def _forward(self, X):
        """Lan truyền tiến (kèm Inverted Dropout khi training)."""
        a_values, z_values = [X], []
        self.dropout_masks = []
        a = X
        num_layers = len(self.W)
        for l in range(num_layers):
            z = self.W[l] @ a + self.b[l]
            a = self._sigmoid(z)
            if self.training and self.dropout_rate > 0.0 and l < num_layers - 1:
                mask = (np.random.rand(*a.shape) >= self.dropout_rate) / (1.0 - self.dropout_rate)
                a = a * mask
                self.dropout_masks.append(mask)
            else:
                self.dropout_masks.append(np.ones_like(a))
            z_values.append(z)
            a_values.append(a)
        return a_values, z_values

    def _backward(self, a_values, y):
        """Lan truyền ngược + gradient L1/L2 + mask Dropout."""
        n_samples = y.shape[1]
        dW, db = [], []
        a_out = a_values[-1]
        delta = (a_out - y) * self._sigmoid_derivative(a_out)
        num_layers = len(self.W)
        for l in reversed(range(num_layers)):
            a_prev = a_values[l]
            dW_l = (1.0 / n_samples) * (delta @ a_prev.T)
            db_l = (1.0 / n_samples) * np.sum(delta, axis=1, keepdims=True)
            # L2 vào gradient (trừ AdamW xử lý tách rời ở update)
            if self.l2_lambda > 0.0 and self.optimizer != "adamw":
                dW_l += self.l2_lambda * self.W[l]
            # L1
            if self.l1_lambda > 0.0:
                dW_l += self.l1_lambda * np.sign(self.W[l])
            dW.insert(0, dW_l)
            db.insert(0, db_l)
            if l > 0:
                delta = (self.W[l].T @ delta) * self._sigmoid_derivative(a_prev)
                if self.training and self.dropout_rate > 0.0:
                    delta = delta * self.dropout_masks[l - 1]
        return dW, db

    def _update_parameters(self, dW, db, t, lr):
        """Cập nhật trọng số theo optimizer đã chọn."""
        for l in range(len(self.W)):
            if self.optimizer == "sgd":
                self.W[l] -= lr * dW[l]; self.b[l] -= lr * db[l]
            elif self.optimizer == "momentum":
                self.v_W[l] = self.beta1 * self.v_W[l] + (1 - self.beta1) * dW[l]
                self.v_b[l] = self.beta1 * self.v_b[l] + (1 - self.beta1) * db[l]
                self.W[l] -= lr * self.v_W[l]; self.b[l] -= lr * self.v_b[l]
            elif self.optimizer == "nesterov":
                self.v_W[l] = self.beta1 * self.v_W[l] + dW[l]
                self.v_b[l] = self.beta1 * self.v_b[l] + db[l]
                self.W[l] -= lr * (dW[l] + self.beta1 * self.v_W[l])
                self.b[l] -= lr * (db[l] + self.beta1 * self.v_b[l])
            elif self.optimizer == "adagrad":
                self.s_W[l] += dW[l] ** 2; self.s_b[l] += db[l] ** 2
                self.W[l] -= lr * dW[l] / (np.sqrt(self.s_W[l]) + self.epsilon)
                self.b[l] -= lr * db[l] / (np.sqrt(self.s_b[l]) + self.epsilon)
            elif self.optimizer == "rmsprop":
                self.s_W[l] = self.beta2 * self.s_W[l] + (1 - self.beta2) * (dW[l] ** 2)
                self.s_b[l] = self.beta2 * self.s_b[l] + (1 - self.beta2) * (db[l] ** 2)
                self.W[l] -= lr * dW[l] / (np.sqrt(self.s_W[l]) + self.epsilon)
                self.b[l] -= lr * db[l] / (np.sqrt(self.s_b[l]) + self.epsilon)
            elif self.optimizer in ("adam", "adamw"):
                self.m_W[l] = self.beta1 * self.m_W[l] + (1 - self.beta1) * dW[l]
                self.m_b[l] = self.beta1 * self.m_b[l] + (1 - self.beta1) * db[l]
                self.v_W[l] = self.beta2 * self.v_W[l] + (1 - self.beta2) * (dW[l] ** 2)
                self.v_b[l] = self.beta2 * self.v_b[l] + (1 - self.beta2) * (db[l] ** 2)
                m_W_hat = self.m_W[l] / (1.0 - self.beta1 ** t)
                m_b_hat = self.m_b[l] / (1.0 - self.beta1 ** t)
                v_W_hat = self.v_W[l] / (1.0 - self.beta2 ** t)
                v_b_hat = self.v_b[l] / (1.0 - self.beta2 ** t)
                if self.optimizer == "adamw" and self.l2_lambda > 0.0:
                    self.W[l] -= lr * self.l2_lambda * self.W[l]   # decoupled weight decay
                self.W[l] -= lr * m_W_hat / (np.sqrt(v_W_hat) + self.epsilon)
                self.b[l] -= lr * m_b_hat / (np.sqrt(v_b_hat) + self.epsilon)

    def _full_loss_acc(self, X_in, y_in):
        """Tính Loss (MSE + penalty) và Accuracy ở chế độ Evaluation (không dropout)."""
        self.training = False
        a_values, _ = self._forward(X_in)
        y_hat = a_values[-1]
        loss = 0.5 * np.mean(np.sum((y_hat - y_in) ** 2, axis=0))
        if self.l2_lambda > 0.0:
            loss += 0.5 * self.l2_lambda * sum(np.sum(w ** 2) for w in self.W)
        if self.l1_lambda > 0.0:
            loss += self.l1_lambda * sum(np.sum(np.abs(w)) for w in self.W)
        acc = float(np.mean((y_hat >= 0.5).astype(int) == y_in))
        return loss, acc

    def fit(self, X, y, X_val=None, y_val=None, augment=False, verbose=False):
        """Huấn luyện Mini-batch + Optimizer + Regularization + Early Stopping."""
        n_samples = X.shape[0]
        y_col = y.reshape(-1, 1) if y.ndim == 1 else y
        X_tr_eval = X.T
        y_tr_eval = y_col.T
        has_val = X_val is not None and y_val is not None
        if has_val:
            X_val_eval = X_val.T
            y_val_eval = (y_val.reshape(-1, 1) if y_val.ndim == 1 else y_val).T

        self.loss_history, self.val_loss_history = [], []
        self.accuracy_history, self.val_accuracy_history = [], []
        best_val_loss, no_improve = float("inf"), 0
        self.best_W = [np.copy(w) for w in self.W]
        self.best_b = [np.copy(bias) for bias in self.b]
        self.best_epoch = 0

        total_steps = self.epochs * int(np.ceil(n_samples / self.batch_size))
        t = 0
        for epoch in range(1, self.epochs + 1):
            idx = np.random.permutation(n_samples)
            X_sh, y_sh = X[idx], y_col[idx]
            for i in range(0, n_samples, self.batch_size):
                t += 1
                X_batch = X_sh[i:i + self.batch_size].T
                y_batch = y_sh[i:i + self.batch_size].T
                # Data Augmentation: thêm nhiễu Gaussian nhỏ vào batch
                if augment:
                    X_batch = X_batch + np.random.normal(0, 0.05, X_batch.shape)
                # Forward (training mode -> có dropout)
                self.training = True
                a_values, _ = self._forward(X_batch)
                dW, db = self._backward(a_values, y_batch)
                # Gradient Clipping
                if self.clip_norm is not None:
                    gnorm = np.sqrt(sum(np.sum(g ** 2) for g in dW) + sum(np.sum(g ** 2) for g in db))
                    if gnorm > self.clip_norm:
                        coef = self.clip_norm / (gnorm + 1e-6)
                        dW = [g * coef for g in dW]; db = [g * coef for g in db]
                # Learning rate: warmup + scheduler
                lr = self.init_lr
                if self.warmup_steps > 0 and t <= self.warmup_steps:
                    lr = self.init_lr * (t / self.warmup_steps)
                elif self.lr_scheduler == "cosine":
                    prog = (t - self.warmup_steps) / max(1, total_steps - self.warmup_steps)
                    lr = self.init_lr * 0.5 * (1.0 + np.cos(np.pi * min(prog, 1.0)))
                elif self.lr_scheduler == "step":
                    lr = self.init_lr * (0.5 ** int((t - self.warmup_steps) / max(1, total_steps * 0.3)))
                self.lr = lr
                self._update_parameters(dW, db, t, lr)

            # Cuối epoch: đánh giá full train & val (eval mode)
            tr_loss, tr_acc = self._full_loss_acc(X_tr_eval, y_tr_eval)
            self.loss_history.append(tr_loss)
            self.accuracy_history.append(tr_acc)
            if has_val:
                val_loss, val_acc = self._full_loss_acc(X_val_eval, y_val_eval)
                self.val_loss_history.append(val_loss)
                self.val_accuracy_history.append(val_acc)
                # Early Stopping theo Val Loss
                if self.patience is not None:
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        self.best_W = [np.copy(w) for w in self.W]
                        self.best_b = [np.copy(bias) for bias in self.b]
                        self.best_epoch = epoch
                        no_improve = 0
                    else:
                        no_improve += 1
                    if no_improve >= self.patience:
                        self.W = [np.copy(w) for w in self.best_W]
                        self.b = [np.copy(bias) for bias in self.best_b]
                        break
            if verbose and (epoch == 1 or epoch % 200 == 0 or epoch == self.epochs):
                msg = f"Epoch {epoch:4d}/{self.epochs} | Train Loss {tr_loss:.4f} | Train Acc {tr_acc:.2%}"
                if has_val:
                    msg += f" | Val Acc {self.val_accuracy_history[-1]:.2%}"
                print(msg)

        if self.patience is None or no_improve < self.patience:
            self.best_W = [np.copy(w) for w in self.W]
            self.best_b = [np.copy(bias) for bias in self.b]
            self.best_epoch = epoch
        return self

    def predict_proba(self, X):
        self.training = False
        a_values, _ = self._forward(X.T)
        return a_values[-1].T

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int).ravel()

print("Đã định nghĩa thành công lớp MLP hợp nhất!")

## Bước 3: Dữ liệu (Train nhỏ + Test lớn để thấy rõ overfitting)

Dùng **một bộ dữ liệu chung** cho cả hai phần:
- `make_moons`, **1500 mẫu**, **nhiễu 0.25** (có quy luật thật để học, vẫn dễ overfit).
- **120 Train / 120 Val / 1260 Test** → Train nhỏ ép overfit, Test lớn để Test Accuracy đủ mịn.

In [ ]:
# Bước 3: Tạo và phân tách dữ liệu
np.random.seed(42)
X, y = make_moons(n_samples=1500, noise=0.25, random_state=42)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, train_size=240, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

layer_sizes = [2, 64, 32, 1]   # mạng lớn so với 120 mẫu train -> dễ overfit
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

# PHẦN A — Optimization: So sánh các Optimizers

**Mục tiêu:** xem optimizer nào học **nhanh và tốt nhất** trên tập Train (tốc độ hội tụ).

Ở phần này ta **TẮT hết regularization** để cô lập đúng tác dụng của optimizer. Mọi cấu hình dùng cùng kiến trúc, cùng mini-batch, và **reset seed** trước mỗi lần train để khởi tạo trọng số giống nhau.

In [ ]:
# Bước 4: Huấn luyện so sánh Optimizers (KHÔNG regularization)
opt_configs = {
    "SGD":      {"optimizer": "sgd",      "lr": 0.30},
    "Momentum": {"optimizer": "momentum", "lr": 0.30},
    "Nesterov": {"optimizer": "nesterov", "lr": 0.10},
    "AdaGrad":  {"optimizer": "adagrad",  "lr": 0.10},
    "RMSprop":  {"optimizer": "rmsprop",  "lr": 0.01},
    "Adam":     {"optimizer": "adam",     "lr": 0.01},
}

opt_epochs = 300
opt_results = {}

print("=" * 60)
print("  PHẦN A: So sánh Optimizers (no regularization)")
print("=" * 60)
for name, cfg in opt_configs.items():
    np.random.seed(42)  # cùng khởi tạo trọng số
    mlp = MLP(
        layer_sizes=layer_sizes, learning_rate=cfg["lr"], epochs=opt_epochs,
        batch_size=32, optimizer=cfg["optimizer"],
    )
    mlp.fit(X_train, y_train, X_val, y_val)
    test_acc = float(np.mean(mlp.predict(X_test) == y_test))
    opt_results[name] = {
        "loss_history": mlp.loss_history,
        "acc_history": mlp.accuracy_history,
        "test_acc": test_acc,
        "final_train_loss": mlp.loss_history[-1],
    }
    print(f"{name:<10} | Final Train Loss: {mlp.loss_history[-1]:.4f} | Test Acc: {test_acc:.2%}")

In [ ]:
# Bước 5: Đồ thị hội tụ Loss & Accuracy của các Optimizers
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
for name, d in opt_results.items():
    plt.plot(d["loss_history"], label=f"{name} (Test {d['test_acc']:.1%})", linewidth=2.0)
plt.xlabel("Epoch"); plt.ylabel("Train Loss (MSE)")
plt.title("PHẦN A — Tốc độ hội tụ Loss theo Optimizer")
plt.grid(True, alpha=0.3); plt.legend()

plt.subplot(1, 2, 2)
for name, d in opt_results.items():
    plt.plot(d["acc_history"], label=name, linewidth=2.0)
plt.xlabel("Epoch"); plt.ylabel("Train Accuracy")
plt.title("PHẦN A — Train Accuracy theo Optimizer")
plt.grid(True, alpha=0.3); plt.legend()

plt.tight_layout()
plt.show()

print("Nhận xét: Adam/RMSprop thường hội tụ NHANH nhất (Loss giảm sớm).")
print("Nhưng học nhanh + không regularization => dễ overfit (xem Phần B).")

# PHẦN B — Overfitting: So sánh DÙNG vs KHÔNG DÙNG Regularization

**Mục tiêu:** xem mỗi kỹ thuật regularization giúp **giảm overfitting** ra sao.

Ta **cố định optimizer = Adam** (ông học nhanh nhất ở Phần A, cũng dễ overfit nhất), rồi:
- **Không dùng** = Adam trắng (không regularization).
- **Dùng** = Adam + **chỉ bật riêng một kỹ thuật**.

Hai cột quan trọng:
- **Δ Test Acc** = Test(Dùng) − Test(Không) → muốn **dương**.
- **Δ Gap** = Gap(Dùng) − Gap(Không), với Gap = Train Acc − Val Acc → muốn **âm**.

In [ ]:
# Bước 6: Train cô lập từng kỹ thuật regularization (optimizer = Adam cố định)
reg_epochs = 400

def train_reg(augment=False, **kwargs):
    np.random.seed(42)  # cùng khởi tạo trọng số -> chỉ khác đúng kỹ thuật
    mlp = MLP(
        layer_sizes=layer_sizes, learning_rate=0.01, epochs=reg_epochs,
        batch_size=32, optimizer="adam", **kwargs
    )
    mlp.fit(X_train, y_train, X_val, y_val, augment=augment)
    e = mlp.best_epoch
    tr_acc = mlp.accuracy_history[e - 1]
    val_acc = mlp.val_accuracy_history[e - 1]
    test_acc = float(np.mean(mlp.predict(X_test) == y_test))
    return {"epochs": e, "train_acc": tr_acc, "val_acc": val_acc,
            "test_acc": test_acc, "gap": tr_acc - val_acc}

reg_techniques = {
    "L1 Regularization": {"l1_lambda": 0.0005},
    "L2 Regularization": {"l2_lambda": 0.01},
    "Dropout":           {"dropout_rate": 0.25},
    "Data Augmentation": {"augment": True},
    "Early Stopping":    {"patience": 30},
}

base = train_reg()   # Adam trắng (không regularization)
reg_pairs = {name: {"off": base, "on": train_reg(**kw)} for name, kw in reg_techniques.items()}
print(f"Baseline Adam (không regularization): Test {base['test_acc']:.2%} | Gap {base['gap']:.2%}")
print("Đã train xong các cấu hình regularization cô lập.")

In [ ]:
# Bước 7: Bảng + biểu đồ DÙNG vs KHÔNG DÙNG regularization
header = (f"| {'Kỹ thuật':<20} | {'Test (Không)':>12} | {'Test (Dùng)':>12} | {'Δ Test':>8} | "
         f"{'Gap (Không)':>11} | {'Gap (Dùng)':>10} | {'Δ Gap':>8} |")
print(header)
print("|" + "-" * (len(header) - 2) + "|")
for name, r in reg_pairs.items():
    off, on = r["off"], r["on"]
    d_test = on["test_acc"] - off["test_acc"]
    d_gap = on["gap"] - off["gap"]
    print(f"| {name:<20} | {off['test_acc']:>11.2%} | {on['test_acc']:>11.2%} | {d_test:>+7.2%} | "
          f"{off['gap']:>10.2%} | {on['gap']:>9.2%} | {d_gap:>+7.2%} |")
print("\nGhi chú: Δ Test dương = tốt hơn | Δ Gap âm = giảm overfitting.")

names = list(reg_pairs.keys())
off_gap = [reg_pairs[n]["off"]["gap"] for n in names]
on_gap = [reg_pairs[n]["on"]["gap"] for n in names]
off_test = [reg_pairs[n]["off"]["test_acc"] for n in names]
on_test = [reg_pairs[n]["on"]["test_acc"] for n in names]
xx = np.arange(len(names)); w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(xx - w/2, off_test, w, label="Không dùng", color="#d95f5f")
axes[0].bar(xx + w/2, on_test, w, label="Dùng", color="#5f9ed9")
axes[0].set_title("PHẦN B — Test Accuracy (càng cao càng tốt)")
axes[0].set_xticks(xx); axes[0].set_xticklabels(names, rotation=20, ha="right")
axes[0].legend(); axes[0].grid(True, axis="y", alpha=0.3)
axes[1].bar(xx - w/2, off_gap, w, label="Không dùng", color="#d95f5f")
axes[1].bar(xx + w/2, on_gap, w, label="Dùng", color="#5f9ed9")
axes[1].set_title("PHẦN B — Overfitting Gap (càng thấp càng tốt)")
axes[1].set_xticks(xx); axes[1].set_xticklabels(names, rotation=20, ha="right")
axes[1].legend(); axes[1].grid(True, axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

# PHẦN C — Kết hợp cả hai trụ cột

So sánh 3 mô hình để thấy vai trò của TỪNG trụ cột và khi GỘP lại:

1. **Baseline thô sơ**: `SGD`, không regularization → học chậm + overfit.
2. **Chỉ Optimizer tốt**: `Adam`, không regularization → học nhanh nhưng vẫn overfit.
3. **Kết hợp đầy đủ**: `AdamW` + L2(decoupled) + Dropout + Augmentation + Early Stopping + Cosine scheduler + Warmup + Gradient Clipping → vừa học tốt vừa tổng quát hóa.

Kèm hình **ranh giới quyết định** của Baseline vs Kết hợp.

In [ ]:
# Bước 8: Huấn luyện 3 mô hình kết hợp
combo_models = {}

# 1) Baseline thô sơ: SGD, không regularization
np.random.seed(42)
m_base = MLP(layer_sizes=layer_sizes, learning_rate=0.30, epochs=400,
             batch_size=32, optimizer="sgd").fit(X_train, y_train, X_val, y_val)
combo_models["1. SGD thô sơ"] = m_base

# 2) Chỉ Optimizer tốt: Adam, không regularization
np.random.seed(42)
m_adam = MLP(layer_sizes=layer_sizes, learning_rate=0.01, epochs=400,
             batch_size=32, optimizer="adam").fit(X_train, y_train, X_val, y_val)
combo_models["2. Adam (no reg)"] = m_adam

# 3) Kết hợp đầy đủ: AdamW + toàn bộ regularization + scheduler/warmup/clipping
np.random.seed(42)
m_full = MLP(
    layer_sizes=layer_sizes, learning_rate=0.01, epochs=400, batch_size=32,
    optimizer="adamw", l2_lambda=0.01, dropout_rate=0.25, patience=30,
    clip_norm=1.0, lr_scheduler="cosine", warmup_steps=50,
).fit(X_train, y_train, X_val, y_val, augment=True)
combo_models["3. AdamW + Full Reg"] = m_full

print(f"| {'Mô hình':<22} | {'Epochs':>6} | {'Train Acc':>9} | {'Val Acc':>8} | {'Test Acc':>8} | {'Gap':>7} |")
print("|" + "-" * 74 + "|")
combo_summary = {}
for name, m in combo_models.items():
    e = m.best_epoch
    tr = m.accuracy_history[e - 1]
    va = m.val_accuracy_history[e - 1]
    te = float(np.mean(m.predict(X_test) == y_test))
    combo_summary[name] = {"test": te, "gap": tr - va}
    print(f"| {name:<22} | {e:>6} | {tr:>8.2%} | {va:>7.2%} | {te:>7.2%} | {tr - va:>6.2%} |")

In [ ]:
# Bước 9: Ranh giới quyết định — Baseline (SGD) vs Kết hợp đầy đủ (AdamW + Full Reg)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx_g, yy_g = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid_scaled = scaler.transform(np.c_[xx_g.ravel(), yy_g.ravel()])
X_test_orig = X_test * scaler.scale_ + scaler.mean_

for ax, (name, m) in zip(axes, [("1. SGD thô sơ", m_base), ("3. AdamW + Full Reg", m_full)]):
    probs = m.predict_proba(grid_scaled).reshape(xx_g.shape)
    ax.contourf(xx_g, yy_g, probs, levels=50, cmap="RdYlBu", alpha=0.6)
    ax.scatter(X_test_orig[:, 0], X_test_orig[:, 1], c=y_test, cmap="RdYlBu",
               edgecolors="black", linewidth=0.5, s=18)
    ax.set_title(f"{name}\nTest Acc: {combo_summary[name]['test']:.2%} | Gap: {combo_summary[name]['gap']:.2%}")
    ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# Kết luận: Optimization & Regularization bổ trợ cho nhau

| | **Optimization** | **Regularization** |
| :--- | :--- | :--- |
| Trả lời câu hỏi | Học **nhanh** & hội tụ tốt? | **Tổng quát hóa** tốt? |
| Tác động chính | Train Loss giảm nhanh | Gap (Train − Val) nhỏ |
| Đo ở đâu | Đường cong Loss (Phần A) | Bảng Gap & Test Acc (Phần B) |
| Ví dụ công cụ | Adam, RMSprop, Momentum, scheduler | L2, Dropout, Early Stopping, Augmentation |

**Ba bài học từ notebook:**
1. **Phần A** — Optimizer tốt (Adam/RMSprop) làm Loss giảm **nhanh hơn** SGD, nhưng *học nhanh không đảm bảo tổng quát hóa*.
2. **Phần B** — Khi cố định Adam, bật regularization **giảm Gap** và thường **tăng Test Acc** (mạnh nhất: L2, Dropout).
3. **Phần C** — Kết hợp **Optimizer tốt + Regularization đầy đủ** cho mô hình **vừa học tốt vừa ít overfit** (ranh giới trơn tru, Gap nhỏ).

> **Key takeaway:** Optimizer quyết định *bạn xuống đáy thung lũng nhanh thế nào*; Regularization quyết định *cái đáy đó có tổng quát hóa ra dữ liệu mới hay không*. Một pipeline huấn luyện tốt cần **cả hai**.

---
⚠️ *Lưu ý: notebook chưa được chạy trên máy này (thiếu Python). Hãy Run All để xem số liệu & biểu đồ thực tế.*